## 环境搭建说明

RISC-V 中国峰会 2026 教程网站：https://tutorial.xiangshan.cc/rvsc26/hands_on/setup/

1. 在浏览器中打开 https://bootcamp.xiangshan.cc
2. 输入密码：**现场公布**
3. 默认会显示终端；如果没有，请点击左上角菜单（三条横线），选择“Terminal > New Terminal”
4. 在终端中运行 `./start.sh`
5. 脚本会生成唯一随机 ID，并在 `/data/${random_id}` 创建工作区
6. 工作区创建后应自动打开；否则点击“Menu > File > Open Folder”，输入 `/data/${random_id}` 并点击“OK”
7. 遇到问题可随时向讲解者求助

## 欢迎参加香山教程

本节介绍 Jupyter Notebook 的基本用法。

Jupyter Notebook 由多个单元格组成，每个单元格可以包含 Python/Bash 代码或文本。文本单元格用于编写说明，代码单元格用于放置可运行的代码。

- 以 `%%bash` 开头的单元格是 Bash 脚本，其余为 Python 代码。
- 以 `#` 开头的行是注释。

点击代码单元格左上角的 ▶ 运行该单元格，输出会显示在下方。首次打开 Notebook 时，运行环境可能无法正确识别。如果运行时页面顶部提示“Select Kernel”，请选择“Python Environments” > “Python 3.x.y /bin/python3 (Global Env)”。

<div style="text-align: center;"><img src="../images/01-basics/00-welcome/select-kernel-1.png" width="25%"> > <img src="../images/01-basics/00-welcome/select-kernel-2.png" width="25%"></div>

In [ ]:
%%bash
echo "[bash] Welcome to the XiangShan Bootcamp!"

In [ ]:
print("[python] Welcome to the XiangShan Bootcamp!")

每个单元格都有自己的工作目录和环境变量，因此某些命令可能需要重新运行。若直接在 Shell 中执行命令，则可跳过重复步骤，例如 `source env.sh`。

还需要说明的是，这与直接在 Shell 中执行命令有所不同。在 Shell 中要留意当前工作目录；设置环境变量时不必重复执行 source env.sh。

In [ ]:
%%bash
# 举个例子，在这个单元格中，我们更改工作目录，...
pwd
cd ..
pwd
# ... 并设置一些环境变量，
export HELLO_WORLD="XiangShan Bootcamp!"
echo Hello $HELLO_WORLD

In [ ]:
%%bash
# ... 我们可以看到前一个单元格中所做的更改在此单元格中不会生效。
pwd
echo Hello $HELLO_WORLD

### 关于 `get_asset` 函数

后续章节会用到 `get_asset` 函数。它可优先使用本地编译产物；若不存在，则回退到预编译资源。


```bash
# 举个例子, 使用预编译的仿真程序 emu
$(get_asset emu-precompile/emu)
# 这将返回 assets/emu-precompile/emu 的绝对路径

# 或者，优先使用本地编译的文件（如果存在）
$(get_asset emu-precompile/emu ${NOOP_HOME}/build/emu)
# 如果你已经进行了本地编译，${NOOP_HOME}/build/emu 将存在并被优先返回。否则，它将回退到 assets/emu-precompile/emu
```

编译和运行香山需要大量计算资源。为使资源较弱的设备也能体验香山，bootcamp 提供了许多预编译资源，我们将通过 get_asset 函数使用这些资源。本教程演示服务器资源有限，因此大多数编译命令和部分运行命令只展示而不实际执行，以节省计算资源和时间。


## 配置与构建

本节介绍香山的配置和构建流程。

### 配置

得益于面向对象硬件描述语言 Chisel 的灵活性，香山提供了大量可配置参数，方便研究人员和工业界用户定制。

```scala
// xiangshan/frontend/FrontendParameters.scala
case class FrontendParameters(
    FetchBlockSize:   Int              = 64, // bytes
    bpuParameters:    BpuParameters    = BpuParameters(),
    icacheParameters: ICacheParameters = ICacheParameters(),
    // ...
) { ... }

// xiangshan/frontend/icache/Parameters.scala
case class ICacheParameters(
    NumSets:   Int     = 256,
    NumWays:   Int     = 4,
    Replacer:  String  = "setplru", // "random", "setlru", "setplru"
    EnableEcc: Boolean = true,      // whether to enable ECC or parity check
    // ...
) { ... }
```

以昆明湖 V3 的前端设计为例，我们采用分层参数系统：设计者首先为各模块定义参数类和默认参数，如本页所示。

随后，研究者和工业界用户可以在 `top/Configs.scala` 中实例化这些参数类，快速生成不同配置。最小配置可减少资源，从而加快编译和仿真，用于快速功能验证。


```scala
// Default configuration
class DefaultConfig extends Config(
    // ...
    frontendParameters = FrontendParameters()
    // ...
)

// Minimal configuration for quick functional verification, with smaller resources to speed up compilation and simulation
class MinimalConfig extends Config(
    // ...
    frontendParameters = FrontendParameters(
      FetchBlockSize = 32,
      icacheParameters = ICacheParameters(
        NumSets = 64,
      )
    )
    // ...
)
```

最后，可在构建时通过 `make CONFIG=XXXConfig` 编译不同配置。

### 稍等！

由于历史原因，香山构建流程依赖某些环境变量，例如 `NOOP_HOME`。我们提供了 `env.sh` 脚本来设置这些环境变量。运行下面的单元格即可查看变量。

In [ ]:
%%bash
source ../env.sh

env | grep _HOME | outputBuffer

### 构建

完成环境设置后，进入 `$NOOP_HOME` 构建香山。

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

#make emu -j16 CONFIG=MinimalConfig

# 其他 make 选项
# CONFIG=MinimalConfig  XiangShan 配置
# EMU_THREADS=4         仿真线程数
# EMU_TRACE=1           启用波形
# WITH_DRAMSIM=1        使用 DRAMSim3 仿真 DRAM
# WITH_CHISELDB=1       启用 ChiselDB
# WITH_CONSTANTIN=1     启用 Constantin

上述命令会生成 `build/emu` 和 `build/rtl` 等产物。

- `build/rtl/*.sv` 是 Chisel 生成的 Verilog 文件。
- `build/emu` 是使用 Verilator 编译得到的仿真可执行文件。

## 使用 Nexus-AM 构建工作负载

上一节构建了香山仿真器 `emu`，但无法直接在其上运行任意 ELF 文件，因为香山是裸机设备，不提供操作系统或运行时库。Linux 太重量级；Nexus-AM 则是面向裸机的轻量级框架。

`am/` 目录包含 Nexus-AM 框架源码；`apps/` 和 `tests/` 存放常见工作负载源码，你也可以创建自己的应用和测试。

In [ ]:
%%bash
source ../env.sh
cd ${AM_HOME}

tree -d -L 1 | outputBuffer

echo "- Apps: $(ls -m ./apps)"
echo "- Tests: $(ls -m ./tests)"

从简单的“Hello, XiangShan”示例开始。源码位于 `${AM_HOME}/apps/hello`，可使用 `make` 编译：

In [ ]:
%%bash
source ../env.sh
cd $AM_HOME/apps/hello

# 编译
# ARCH=riscv64-xs 代表 XiangShan
# LINUX_GNU_TOOLCHAIN=1 使用 Ubuntu apt 仓库中的 GNU 工具链，而不是 riscv64-unknown-elf
make ARCH=riscv64-xs LINUX_GNU_TOOLCHAIN=1 | outputBuffer

# 检查输出
ls build

编译会生成三个文件：

- `hello-riscv64-xs.bin`：供 emu 使用的程序二进制镜像（已移除 ELF 文件头和其他元数据）。
- `hello-riscv64-xs.elf`：程序的 ELF 文件。
- `hello-riscv64-xs.txt`：程序反汇编文件，用于调试。

Nexus-AM 支持多种 ISA 和配置。传入不同的 `ARCH` 值即可面向不同目标编译。例如，`riscv64-xs` 对应香山的 64 位 RISC-V 架构。如果 ARCH 不受支持，`make` 会列出所有支持的值。

In [ ]:
%%bash
source ../env.sh
cd ${AM_HOME}

# || true 来防止 notebook 因非零退出码而报错
make ARCH= || true

## 运行 RTL 仿真

现在可以使用刚才编译的 `emu` 和 `hello` 程序运行香山 RTL 仿真了！

香山 emu 支持许多选项，运行 `emu --help` 查看用法。

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

$(get_asset emu-precompile/emu ${NOOP_HOME}/build/emu) --help | outputBuffer

运行 Hello XiangShan：

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

$(get_asset emu-precompile/emu ${NOOP_HOME}/build/emu) \
    -i $(get_asset workload/hello-riscv64-xs.bin ${AM_HOME}/apps/hello/build/hello-riscv64-xs.bin) \
    --no-diff \
    2>/dev/null | outputBuffer

注意：运行结束时，香山会将性能计数器输出到 stderr，内容很多。建议始终将 stderr 重定向到文件。上例中不需要计数器，因此重定向到 `/dev/null`，可按需调整。

![MinJie Cycle](../images/02-functional/00-intro/overal-en.png)

我们使用 MinJie 平台搭建工作流。功能验证可概括为四部分组成的循环，每部分都有对应工具支持：


- 测试生成：使用 nexus-am 生成裸机测试；
- 错误检测：使用 nemu 提供黄金参考结果，使用 difftest 比较结果；
- 保留错误现场：使用 lightsss 自动捕获现场；
- 问题排查与修复：使用波形和 chiselDB 定位并修复错误。

### 为什么需要 MinJie？

我们手动在 ALU 模块中注入了一个错误，并编译了 emu 供你测试：

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

$(get_asset emu-precompile/emu-alu-err) \
    -i $(get_asset workload/hello-riscv64-xs.bin ${AM_HOME}/apps/hello/build/hello-riscv64-xs.bin) \
    --no-diff \
    -C 20000 \
    2>/dev/null || true

它没有像上一节那样正确打印“Hello XiangShan”。达到 20,000 周期上限时，`pc` 为 `0xE`，显然不符合正常程序行为。仿真程序无法自行判断运行是否正确，因此错误发生后可能继续运行并浪费资源。

- 仿真程序如何知道已经发生错误？
- 如何定位错误发生的位置？

这是处理器功能验证中的重要问题，MinJie 工具链旨在解决这些问题。

## NEMU：ISA 参考模型

“仿真程序如何知道已经发生错误？”

→ 使用参考模型

NEMU：类似 Spike 的 ISA 模拟器

- 具备类似 QEMU 的性能；
- 提供 API，可比较并验证香山的体系结构状态。

请回忆我们之前提出的问题，我们需要定义什么是“正确”，也就是需要黄金参考模型。QEMU 和 Spike 可以胜任，但开销稍大，也不易集成到工作流中。

→ NEMU

NEMU 提供两种默认配置：
- `xxx_defconfig`：独立运行模式的默认设置。
- `xxx-ref_defconfig`：DiffTest 协同仿真模式的默认设置。

稍后会介绍 difftest 和参考模式。先看看独立模式的构建步骤和用法。

In [ ]:
%%bash
source ../env.sh
cd ${NEMU_HOME}

make clean

# 编译默认配置为独立模式
make riscv64-xs_defconfig | outputBuffer
make -j | outputBuffer

make clean-softfloat | outputBuffer

# 编译默认配置为参考模式
make riscv64-xs-ref_defconfig | outputBuffer
make -j | outputBuffer

接下来，在 NEMU 独立模式下运行 Hello XiangShan。

In [ ]:
%%bash
source ../env.sh
cd ${NEMU_HOME}

# 使用 -b 选项以批处理模式启动 NEMU，避免手动输入命令来运行工作负载。
./build/riscv64-nemu-interpreter \
    -b \
    $(get_asset workload/hello-riscv64-xs.bin) | outputBuffer

## Difftest：ISA 协同仿真框架

- 仿真程序如何知道已经发生错误？
- 如何定位错误发生的位置？

→ 对 RTL（DUT）和参考模型（REF）进行协同仿真

<img src="../images/02-functional/02-difftest/difftest-arch-en.png" width="80%"/>

现在 NEMU 作为参考模型，部分解决了第一个问题：我们知道了“正确”是什么。还需要一种机制，在错误发生时通知 emu 进程或开发者。通过同时运行 RTL 仿真和参考模型并实时比较体系结构状态，可以发现仿真程序中的错误。

运行 emu 时，通过 `--diff <path/to/ref.so>` 指定参考模型动态链接库路径，即可启用 DiffTest。下面重新运行有错误的 emu 并启用 diff：

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

$(get_asset emu-precompile/emu-alu-err) \
    -i $(get_asset workload/hello-riscv64-xs.bin) \
    --diff $(get_asset emu-precompile/riscv64-nemu-interpreter-so) \
    2>/dev/null | tee emu_err.log > /dev/null || true # tutorial：使用 "|| true" 来防止 notebook 因非零退出码而报错；在实际使用中不需要。

tail -n 7 emu_err.log

在 PC `0x0080000078` 处，REF 和 DUT 状态不匹配：REF 中 a0 为 `0x2000`，DUT 中为 `0`。

检测到错误时，Difftest 还会打印有用信息，例如整数、浮点、向量和 CSR 等体系结构寄存器的值。

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

tail -n 95 emu_err.log | head -n 19 | outputBuffer

以及指令提交顺序和其他信息。

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

tail -n 124 emu_err.log | head -n 11 | outputBuffer

Difftest 检测到错误后，可以重新运行仿真，并在报告的失败周期附近启用波形输出。

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

mkdir -p build
rm -f ./build/*.vcd

$(get_asset emu-precompile/emu-alu-err) \
    -i $(get_asset workload/hello-riscv64-xs.bin) \
    --diff $(get_asset emu-precompile/riscv64-nemu-interpreter-so) \
    -b 8000 \
    -e 10000 \
    --dump-wave \
    2>/dev/null >/dev/null || true

echo -n "Dump wave: "
realpath ./build/*.vcd

## LightSSS：轻量级仿真快照系统

- 如何定位错误？→ Difftest
- 如何高效获取波形？
  - 一开始就启用波形 → 占用大量磁盘空间
  - 手动重新运行 → 效率低

借助 DiffTest，我们已经解决了如何定位错误的问题。但找到故障点后，仍需手动启用波形并重新运行仿真，获取调试所需文件。对于 SPEC CPU 等长时间运行的工作负载，这非常耗时。我们需要在检测到错误时自动恢复到错误前状态并生成波形，也就是快照机制。

![LightSSS](../images/02-functional/03-lightSSS/lightSSS-overall-en.png)

LightSSS 利用 Linux 的 `fork()` 系统调用和写时复制机制，以较低开销创建仿真快照。

使用 LightSSS 很简单：运行 emu 时添加 `--enable-fork` 参数即可启用。

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

mkdir -p build
rm -f ./build/*.vcd

$(get_asset emu-precompile/emu-alu-err) \
    -i $(get_asset workload/hello-riscv64-xs.bin) \
    --diff $(get_asset emu-precompile/riscv64-nemu-interpreter-so) \
    --enable-fork \
    2> /dev/null | outputBuffer || true

echo -n "Dump wave: "
realpath ./build/*.vcd

如果看到“the oldest checkpoint start to dump wave and dump nemu log...”，说明 LightSSS 已启动。仿真随后会从最近的快照重新开始并记录波形。

## ChiselDB：便于调试的结构化数据库

<img src="../images/02-functional/04-chiseldb/chiseldb-en.png" alt="chiselDB-overview" style="float:right; width:40%; margin-left:5px;">

对于某些信号（例如总线事务），波形并非最佳选择：
- 存储开销大
- 缺少结构化信息

→ ChiselDB

LightSSS 很有用，但波形文件仍然很大，也难以用于后续分析。我们还希望分析内存事务跟踪等结构化数据，因此推出 ChiselDB。它会在硬件模块接口间插入探针，并通过 Chisel 代码中的 DPI-C 传递 Bundle 信息和数据。错误分析支持 SQL 查询，比查看波形更方便。

我们提供预构建仿真器 `emu-cdb-err`，其中注入一个错误，会将所有从 L2 Cache 发送到 L3 Cache 的数据强制设为常量。使用 `--dump-db` 启用 ChiselDB 并开启 DiffTest；运行后 DiffTest 会报告错误，并在 `./build` 下生成 `.db` 文件。

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

rm -f ./build/*.db # 清除旧文件
mkdir -p build

$(get_asset emu-precompile/emu-cdb-err) \
    -i $(get_asset workload/stream_100000.bin) \
    --diff $(get_asset emu-precompile/riscv64-nemu-interpreter-so) \
    --dump-db \
    2>linux.err || true

echo -n "Dump DB: "
realpath ./build/*.db

使用 SQLite 读取 `.db` 进行分析：查询地址 `0x80048f00` 上的所有 TileLink 事务，并用 `./scripts/cache/convert_tllog.sh` 格式化输出。

In [ ]:
%%bash
source ../env.sh

DB=$(ls -t ${NOOP_HOME}/build/*db | head -n 1)

sqlite3 ${DB} "select * from TLLog where ADDRESS=0x80048f00" | sh ${NOOP_HOME}/scripts/cache/convert_tllog.sh | outputBuffer

列：时间戳、to_from、通道、操作码、权限、地址、数据。

可以看到：

1. L1D 将数据写回 L2。
```
16171 L2_L1D_0 C ProbeAckData Shrink TtoN 0 5 80048f00 0000000080048f50 0000000080014328 0000000000000000 0000000000000000 user: 0 echo: 0
16172 L2_L1D_0 C ProbeAckData Shrink TtoN 0 5 80048f00 0000000000000000 0000000000000000 000000008001e000 0000000080042060 user: 0 echo: 0
```

接着，数据被写回 L3。
```
16179 L3_L2_0 C ProbeAckData Shrink TtoN 0 2 80048f00 0000000080048f50 0000000080014328 0000000000000000 0000000000000000 user: 0 echo: 1
16180 L3_L2_0 C ProbeAckData Shrink TtoN 0 2 80048f00 0000000000000000 0000000000000000 000000008001e000 0000000080042060 user: 0 echo: 1
```

3. L1D 再次请求数据时，L3→L2 的 GrantData 与 L1D 原先写入的数据不一致，说明存在错误。
```
16457 L2_L1D_0 A AcquireBlock Grow NtoT 0 0 80048f00 0000000000000000 0000000000000000 0000000000000000 0000000000000000 user: 80048f07 echo: 0 
16463 L3_L2_0 A AcquireBlock Grow NtoT 0 0 80048f00 0000000000000000 0000000000000000 0000000000000000 0000000000000000 user: 0 echo: 1 
16486 L3_L2_0 D GrantData Cap toT 1 0 80048f00 0000000000abcdef 0000000000000000 0000000000000000 0000000000000000 user: 0 echo: 1 
```


![TL-TEST](../images/02-functional/05-tl_test/tl-test-overall-en.png)


缓存系统与上游模块进行协同验证十分复杂，会妨碍快速迭代。为此我们开发 TL-Test：一个单元级缓存系统验证框架，支持 TileLink 协议、缓存一致性检查和随机测试用例生成。

下面举例说明 TL-Test 如何检测缓存一致性违规。我们注入一个错误，使 grant 数据发生错误移位。TL-Test 生成随机测试并定位缓存设计中特定地址上的传输问题，记录总线事务，再用 `grep` 提取日志分析。

In [ ]:
%%bash
source ../env.sh

cat $(get_asset tltest-precompile/tlt_err.patch) | outputBuffer

In [ ]:
%%bash
source ../env.sh
# cd $TLT_HOME && make coupledL2-test-l2l3-v3 run THREADS_BUILD=16 CXX_COMPILER=clang++-17
# cd $TLT_HOME/run && ./tltest_v3lt 2>&1 | tee tltest_v3lt.log

get_asset tltest-precompile/tltest_err
mkdir -p ${WORK_DIR}/02-functional/05-tltest
cd ${WORK_DIR}/02-functional/05-tltest

cp -r $(get_asset tltest-precompile) ./ && cd ./tltest-precompile
./tltest_err 2>&1 | tee tltest_v3lt.log > /dev/null

tail -n 50 tltest_v3lt.log | head -n 15

错误地址：`0x300`

In [ ]:
%%bash
source ../env.sh
# grep "addr: 0x300" $TLT_HOME/run/tltest_v3lt.log

cd ${WORK_DIR}/02-functional/05-tltest/tltest-precompile && grep "addr: 0x300," tltest_v3lt.log | head -n 10

列：[时间] [日志级别] #节点索引 核心 [通道] [操作码] 来源, 地址, 别名, 数据

L1D 再次获取地址 `0x300` 的独占权限，但 L2 返回的数据与参考模型不一致。
```
[2312] [tl-test-new-INFO] #0 L2[0].C[0] [fire A] [AcquireBlock NtoT] source: 0xc, addr: 0x300, alias: 0

[2498] [tl-test-new-INFO] #0 L2[0].C[0] [fire D] [GrantData toT] source: 0xc, addr: 0x300, alias: 0, data: [ 00 43 ... ]
[2500] [tl-test-new-INFO] #0 L2[0].C[0] [fire D] [GrantData toT] source: 0xc, addr: 0x300, alias: 0, data: [ 00 5d ... ]
```

![MinJie Cycle](../images/04-performance/00-intro/overview-en.png)

除了功能验证，性能验证与优化也是处理器开发的重要环节。本章将介绍团队使用的 MinJie（敏捷）性能验证方法并进行演示。性能验证同样是迭代循环：实现 RTL、运行测试、评估并分析性能。接下来介绍相关工具和优化流程，包括 SimPoint、XSPerf、Top-Down 和 Constantin。

![intro](../images/04-performance/01-checkpoint/intro-en.png)

先从检查点说起。性能评估通常通过仿真（即利用 Verilator 进行软件仿真，或利用 FPGA、硬件仿真器进行硬件加速仿真）运行基准测试，但软件仿真速度慢，FPGA 片上资源有限，仿真器对学术机构而言也过于昂贵。目前已经有工作尝试加速软件仿真或提升 FPGA 易用性，并取得了成效。

但是，我们认为还有一种更简单的方法：检查点。

![method](../images/04-performance/01-checkpoint/method-en.png)

检查点技术简单来说，就是选取程序执行过程中的某些片段，并在这些片段的起始处保存架构状态（即寄存器和内存数据）。后续进行性能评估时，只需加载这些保存的状态，即可直接从该点开始仿真。

这种方法主要带来两大益处：

1. 减少了需要仿真的指令数量。我们无需从头运行整个程序，而只需运行其中的特定片段。
2. 同一程序的不同片段可以并行仿真，从而提高了仿真的并行度。

通过对各片段采集的性能数据进行加权平均，即可估算出程序的整体性能。

这一页幻灯片展示了两种常见的片段选取方法：
1. 均匀采样（Uniform sampling）：即每隔固定数量的指令选取一个片段；
2. SimPoint 采样（SimPoint sampling）：即通过性能分析（profiling）选取能够代表程序整体行为特征的片段。

接下来，我们将演示 SimPoint 如何对程序进行分析、生成检查点，以及如何利用这些检查点进行仿真。

本节使用的部分路径和常量与 `../env.sh` 不同。我们准备了 `01-env.sh` 设置环境变量。运行下面单元格即可查看。

In [ ]:
%%bash
source ../04-performance/01-env.sh

env | grep WORKLOAD= # 需要模拟 / 分析 / 检查点的工作负载
env | grep CHECKPOINT_INTERVAL=
env | grep NEMU=
env | grep _HOME | tail
env | grep _PATH | tail

检查点流程的第一步是编译 SimPoint 工具和 NEMU（检查点模式），并生成检查点恢复程序。

In [ ]:
%%bash
source ../04-performance/01-env.sh

cd ${NEMU_HOME}
git submodule update --init

# 编译 simpoint 生成器
cd ${NEMU_HOME}/resource/simpoint/simpoint_repo
make clean
make

# 编译 NEMU（检查点模式）
cd ${NEMU_HOME}
make clean
make riscv64-xs-cpt_defconfig
make -j8

# 为 ${WORKLOAD} 生成检查点恢复器
cd ${NEMU_HOME}/resource/gcpt_restore
rm -rf ${GCPT_PATH}
make -C ${NEMU_HOME}/resource/gcpt_restore/ \
    O=${GCPT_PATH} \
    GCPT_PAYLOAD_PATH=$(get_asset workload/${WORKLOAD}.bin) \
    CROSS_COMPILE=riscv64-linux-gnu-

SimPoint 用于选取代表性片段。NEMU 用于分析程序并生成检查点。恢复程序类似引导加载程序，会将模拟闪存中保存的内存加载到主存并恢复寄存器。

接下来，我们需要使用 NEMU 运行待设置检查点的程序，以收集程序行为数据，从而进行性能分析。

In [ ]:
%%bash
source ../04-performance/01-env.sh

rm -rf ${RESULT_PATH}

_LOG_PATH=${LOG_PATH}/profiling
mkdir -p ${_LOG_PATH}

${NEMU} ${GCPT} \
    -w ${WORKLOAD} \
    -D ${RESULT_PATH} \
    -C profiling \
    -b \
    --simpoint-profile \
    --cpt-interval ${CHECKPOINT_INTERVAL} \
    > >(tee ${_LOG_PATH}/${WORKLOAD}-out.txt | outputBuffer) 2> >(tee ${_LOG_PATH}/${WORKLOAD}-err.txt)

随后，利用 SimPoint 对收集到的程序行为进行聚类分析，并选取相应的片段。

In [ ]:
%%bash
source ../04-performance/01-env.sh

CLUSTER=${RESULT_PATH}/cluster/${WORKLOAD}
mkdir -p ${CLUSTER}

random1=`head -20 /dev/urandom | cksum | cut -c 1-6`
random2=`head -20 /dev/urandom | cksum | cut -c 1-6`

_LOG_PATH=${LOG_PATH}/cluster
mkdir -p ${_LOG_PATH}

${SIMPOINT} \
    -loadFVFile ${PROFILING_RESULT_PATH}/${WORKLOAD}/simpoint_bbv.gz \
    -saveSimpoints ${CLUSTER}/simpoints0 \
    -saveSimpointWeights ${CLUSTER}/weights0 \
    -inputVectorsGzipped \
    -maxK 3 \
    -numInitSeeds 2 \
    -iters 1000 \
    -seedkm ${random1} \
    -seedproj ${random2} \
    > >(tee ${_LOG_PATH}/${WORKLOAD}-out.txt | outputBuffer) 2> >(tee ${_LOG_PATH}/${WORKLOAD}-err.txt) 

最后，使用 NEMU 重新运行需要进行检查点保存的程序，生成检查点文件。

In [ ]:
%%bash
source ../04-performance/01-env.sh

CLUSTER=${RESULT_PATH}/cluster
_LOG_PATH=${LOG_PATH}/checkpoint
mkdir -p ${_LOG_PATH}

${NEMU} ${GCPT} \
    -w ${WORKLOAD} \
    -D ${RESULT_PATH} \
    -C checkpoint \
    -b \
    -S ${CLUSTER} \
    --cpt-interval ${CHECKPOINT_INTERVAL} \
    > >(tee ${_LOG_PATH}/${WORKLOAD}-out.txt | outputBuffer) 2> >(tee ${_LOG_PATH}/${WORKLOAD}-err.txt)


进入目录 `${RESULT_PATH}/checkpoint/${WORKLOAD}`，可以看到生成的检查点文件；这些文件共有与集群数量相等的 `.gz` 文件，且文件名中注明了检查点的权重。

In [ ]:
%%bash
source ../04-performance/01-env.sh

find "${RESULT_PATH}/checkpoint/${WORKLOAD}" -type f -name "*_.gz" | tail

可以使用 `emu` 运行其中一个生成的检查点，查看效果。

In [ ]:
%%bash
source ../04-performance/01-env.sh

CHECKPOINT=$(find ${RESULT_PATH}/checkpoint/${WORKLOAD} -type f -name "*_.gz" | tail -1)

$(get_asset emu-precompile/emu) \
    -i ${CHECKPOINT} \
    --diff $(get_asset emu-precompile/riscv64-nemu-interpreter-so) \
    --max-cycles=50000 \
    2>/dev/null \
    | outputBuffer

当 `emu` 检测到 gzip 压缩检查点时，会自动解压，并恢复检查点的内存状态和体系结构状态。

## 香山中的性能计数器

用途：收集性能事件以供分析和调优。XSPerf 包含累加型、直方图型和滚动型计数器。


XSPerf:
- 累加型（Accumulate）:
  ```c
  if (valid)
    counter += diff;
  ```
- 直方图型（Histogram）:
  ```c
  if (valid)
    distribution[value / step] += 1;
  ```
- 滚动型（Rolling）:
  ```c
  if (valid)
    counters[segment] += diff;
  if (cycles++ == segment_size) {
    cycles = 0;
    segment++;
  }
  ```

在运行基准测试（benchmark）时，我们需要收集并记录硬件行为（性能事件），以便进行分析和调优。

在 XiangShan RTL 中，我们实现了三种类型的性能计数器，其伪代码如上所示：

- 累加型（Accumulate）：基础计数器，每当发生性能事件时即进行累加；
- 直方图型（Histogram）：记录性能事件发生时的数值分布情况；
- 滚动型（Rolling）：类似于分段式的累加型计数器，用于追踪整个运行过程中各时间段内性能事件数量的变化。

### 累加型与直方图型

### 累加器 (Accumulate) 与直方图 (Histogram)

这两种类型的性能计数器会在仿真结束时输出到标准错误流 (stderr)。

示例（累加器）：已提交指令的总数

```scala
def ifCommitReg(counter: UInt): UInt = Mux(isCommitReg, counter, 0.U)
XSPerfAccumulate("commitInstr", ifCommitReg(trueCommitCnt), XSPerfLevel.CRITICAL)
```

示例（直方图）：L2 缓存获取延迟的分布

```scala
XSPerfHistogram("acquire_period", acquire_period, acquire_period_en, 0, 30, 1, true, true)
XSPerfHistogram("acquire_period", acquire_period, acquire_period_en, 30, 100, 5, true, true)
XSPerfHistogram("acquire_period", acquire_period, acquire_period_en, 100, 200, 10, true, true)
```

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

$(get_asset emu-precompile/emu) \
    -i $(get_asset workload/hello-riscv64-xs.bin) \
    --no-diff 2>stderr.log | tail

echo "=== Last 10 lines:"
tail -n 10 stderr.log

echo "=== Example of XSPerfAccumulate: rob commitInstr"
grep -n "rob: commitInstr," stderr.log | tail

echo "=== Example of XSHistogram: l2cache acquire period"
grep -n "l2cache.slices_0.mshrCtl: acquire_period" stderr.log | tail

运行此单元格查看 XSPerf 示例。

### 滚动型

![rolling](../images/04-performance/02-xsperf/rolling-en.png)

此类计数器使用 ChiselDB 框架，将采集数据存储到 SQLite3 数据库。启用 RollingDB 时，编译需指定 `WITH_ROLLINGDB=1`，运行时需使用 `--dump-db`。

前两类计数器无法反映程序不同片段的特征差异，可能遗漏微架构修改对关键区域的影响，因此需要滚动分析。

⚠️注意：如果你正在教程演示服务器上阅读此 Notebook，请不要重新编译香山，因为这会耗费大量时间和计算资源。

In [ ]:
%%bash
source ../env.sh
cd ${NOOP_HOME}

mkdir -p ${WORK_DIR}/04-performance/02-xsperf

# 教程用：将预生成的 rolling db 复制到教程目录
cp $(get_asset emu-perf-result/xs-perf-rolling.db) \
    ${WORK_DIR}/04-performance/02-xsperf/xs-perf-rolling.db

在这里，我们只需将预先生成的滚动数据库文件复制到工作区即可。你可以在当前 code-server 中查看生成它所用的命令。

获取数据库文件后，我们使用 Python 脚本对其进行分析。下面我们使用 `rollingplot.py` 脚本绘制 IPC 数据。

在香山的 RTL 代码中收集 IPC 数据的方式如下：

```scala
// 每 1000 个周期
XSPerfRolling("ipc", ifCommitReg(trueCommitCnt), 1000, clock, reset)
```

In [ ]:
%%bash
source ../env.sh
cd ${WORK_DIR}/04-performance/02-xsperf

# 使用 Python 脚本分析 rolling db，例如绘制 ipc
python3 ${NOOP_HOME}/scripts/rolling.py plot \
    ./xs-perf-rolling.db \
    --perf-name ipc \
    --output results/perf.png # 输出结果

ls -lh ${WORK_DIR}/04-performance/02-xsperf/results/perf.png

In [ ]:
from IPython.display import display, Image
display(Image(filename="../work/04-performance/02-xsperf/results/perf.png"))

脚本会输出结果图片，展示香山运行该程序时 IPC 随时间的变化：

预期结果：

<img src="../images/04-performance/02-xsperf/result-example.png" alt="Result Example" width="70%" />

## Top-Down

用途：以层次化形式组织性能事件。

[1]: Yasin A. A top-down method for performance analysis and counters architecture\[C\]//2014 IEEE International Symposium on Performance Analysis of Systems and Software (ISPASS). IEEE, 2014: 35-44.

<img src="../images/04-performance/03-topdown/example-en.png" alt="Example" width="70%" />

Top-Down 是一种常用性能分析方法，它将零散的性能事件组织成层次结构，从而更准确地分析各项性能事件对处理器整体性能的影响。

基于“02-xsperf”章节中介绍的 XSPerfAccumulate 机制，我们在 RTL 中实现了一套针对香山微架构和 RISC-V 指令集优化的 Top-Down 计数器，以更好地建模香山并与 XS-GEM5 对齐。

`${NOOP_HOME}/scripts/top-down` 目录下提供了分析脚本，可提取 Top-Down 结果并绘制图表。

⚠️注意：演示服务器上的读者请勿运行这些脚本，因为它们会大量访问磁盘。

In [ ]:
%%bash
source ../env.sh

mkdir -p ${WORK_DIR}/04-performance/03-topdown
cd ${WORK_DIR}/04-performance/03-topdown

# tutorial：复制分析结果
cp -r $(get_asset emu-spec-topdown-result/results) ./

echo === results ===
ls ./results

echo === first 10 lines of results.csv ===
head -n 10 ./results/results.csv

echo === first 10 lines of results-weighted.csv ===
head -n 10 ./results/results-weighted.csv

In [ ]:
from IPython.display import display, Image
display(Image(filename="../work/04-performance/03-topdown/results/result.png"))

脚本会输出图片如下：

预期结果：

<img src="../images/04-performance/03-topdown/result-example.png" alt="Result Example" width="70%" />

## Constantin

用途：通过减少重新编译来加快设计空间探索。

<img src="../images/04-performance/04-constantin/overview-en.png" alt="Overview" width="80%" />

有时需要测试不同参数下的性能。周期精确模拟器可能不完全准确，因此有时要直接在 RTL 上测试。但每次调整少数参数都重新编译非常耗时。Constantin 基于 DPI-C 接口，利用 C++ 函数和 Chisel BlackBox 机制，在运行时初始化阶段配置参数。

使用 Constantin 替换 Scala 参数的大致方式如下：

```scala
/* *** w/o Constantin *** */
val enableSomeModule = WireInit(false.B) // change to true.B, re-compile and re-run

/* *** w/ Constantin *** */
// in RTL
val enableSomeModule = WireInit(Constantin.createRecord("enableSomeModule", initValue = false))
// in constantin.txt
enableSomeModule 0 // change to 1, re-run
```

要启用 Constantin，需要在编译 emu 时使用 `WITH_CONSTANTIN=1` 选项。目前它不是 emu 命令行参数，配置文件必须位于 `${NOOP_HOME}/build/constantin.txt`。

下面的示例使用 Constantin 控制分支预测器开关。运行此单元格可比较开启和关闭时的差异。

In [ ]:
%%bash
source ../env.sh

mkdir -p ${NOOP_HOME}/build

# 使用默认参数运行
rm -f ${NOOP_HOME}/build/constantin.txt || true
$(get_asset emu-precompile/emu-constantin) \
    -i $(get_asset workload/coremark-2-iteration.bin) \
    -C 10000 \
    --no-diff \
    2>/dev/null \
    | tail

#  关闭 Bpu 运行（回退到静态不预测）
echo "constCtrl 1" > ${NOOP_HOME}/build/constantin.txt
$(get_asset emu-precompile/emu-constantin) \
    -i $(get_asset workload/coremark-2-iteration.bin) \
    -C 10000 \
    --no-diff \
    2>/dev/null \
    | tail